# Detection improvement experiments

Task: ML-IMPROVE-001 | Owner: Team lead / Codex | Date: 2026-09-22
Dataset: COCO Car Damage v1 from Notebook 00. Environment: Colab GPU. Status: NOT EXECUTED.
Question: does reducing destructive augmentation improve YOLOv8n localisation on this very small dataset?
Run generic damage first, then five damaged-part classes. New runs preserve existing models.
Only annotated train/validation splits are used. The eight unannotated test images cannot support mAP, precision or recall.

In [ ]:
from pathlib import Path
import sys, json, platform, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

roots = [Path.cwd(), *Path.cwd().parents, Path('/content/NPN-Car-Insurance')]
ROOT = next((p for p in roots if (p/'ml/src/claimvision_ml').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run Notebook 00 first in this Colab runtime.')
sys.path.insert(0, str(ROOT/'ml/src'))
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print({'python': platform.python_version(), 'torch': str(torch.__version__),
       'cuda': torch.cuda.is_available(), 'seed': 42})
if not torch.cuda.is_available():
    raise RuntimeError('Choose a Colab GPU runtime, then run Notebook 00 again.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from claimvision_ml.detection.experiments import prepare_dataset, validate_training_yaml, run_experiment
from PIL import Image
import matplotlib.patches as patches
import yaml
DATA_VERSION = 'v2'
data_paths = {}
for task in ['damage','parts']:
    path = ROOT/'artifacts/runs/improvements'/f'yolo_{task}_data_{DATA_VERSION}'/'data.yaml'
    if not path.exists(): path = prepare_dataset(ROOT, task, DATA_VERSION)
    config, provenance = validate_training_yaml(path)
    data_paths[task] = path
    print(task, {s: {'images': v['image_count'], 'boxes': v['box_count']} for s,v in provenance['splits'].items()})
    fig, axes = plt.subplots(1,3,figsize=(14,4))
    images = sorted((path.parent/'train/images').iterdir())[:3]
    for ax, image_path in zip(axes,images):
        image = Image.open(image_path); ax.imshow(image); ax.axis('off')
        for line in (path.parent/'train/labels'/f'{image_path.stem}.txt').read_text().splitlines():
            cls,cx,cy,w,h = map(float,line.split()); iw,ih = image.size
            x,y = (cx-w/2)*iw,(cy-h/2)*ih
            ax.add_patch(patches.Rectangle((x,y),w*iw,h*ih,fill=False,color='lime'))
            ax.text(x,y,config['names'][int(cls)],color='yellow')
    fig.suptitle(task + ': inspect converted ground-truth boxes before training')
    plt.show()

## Train controlled comparisons

Inspect the grids above before running this cell. Stop if boxes or class names are wrong.
Baseline and candidate both use YOLOv8n, 640 pixels, AdamW, 100 maximum epochs and patience 20. Candidate: mosaic 0.2, no mixup, rotation 5 degrees, translation 0.05, scale 0.2, gentle colour jitter. These are hypotheses, not proven improvements.
Best checkpoints use the installed Ultralytics validation fitness rule; compare runs using validation mAP50–95 plus recall/per-class AP. Save the installed version and training args. Repeat finalists with additional seeds before promotion.

In [ ]:
RUN_PREFIX = 'det-v2'
SEEDS = [42]
records = []
for task,path in data_paths.items():
    for variant in ['baseline','conservative']:
        for seed in SEEDS:
            run_id = f'{RUN_PREFIX}-{task}-{variant}-s{seed}'
            saved = ROOT/'artifacts/runs/improvements'/run_id/'experiment.json'
            if saved.exists():
                record = json.loads(saved.read_text())
                _, current = validate_training_yaml(path)
                if record['status'] != 'COMPLETE_VALIDATION_ONLY' or record['dataset'] != current:
                    raise RuntimeError('Use a new RUN_PREFIX for incomplete/changed runs: '+run_id)
            else:
                record = run_experiment(ROOT,path,run_id,variant,seed=seed)
            records.append(record)
display(pd.DataFrame([{'run':r['run_id'],**r['validation'],
                      'cpu_ms':r['cpu_end_to_end_median_ms']} for r in records]))

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as DisplayImage
for r in records:
    directory = ROOT/'artifacts/runs/improvements'/r['run_id']
    for plot in ['results.png','confusion_matrix.png']:
        candidates = list(directory.rglob(plot))
        if candidates: display(DisplayImage(filename=str(candidates[0]),width=700))
    task = r['dataset']['task']
    config = yaml.safe_load(data_paths[task].read_text())
    model = YOLO(str(ROOT/r['checkpoint']))
    paths = sorted((Path(config['path'])/'val/images').iterdir())[:3]
    fig,axes = plt.subplots(1,3,figsize=(14,4))
    for ax,p in zip(axes,paths):
        result = model.predict(str(p),verbose=False)[0]
        ax.imshow(result.plot()[...,::-1]); ax.axis('off'); ax.set_title(p.name)
    fig.suptitle(r['run_id'] + ': validation predictions (inspect misses/false alarms)')
    plt.show()

In [ ]:
import shutil
bundle = shutil.make_archive(str(ROOT/'detection_improvement_results'),'zip',ROOT/'artifacts/runs','improvements')
print(bundle)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(bundle)

## Findings, limitations and next decision

After running, record whether validation mAP50–95, recall and per-class AP improved. Retain baseline weights if the candidate regresses. The part model remains experimental pending class-wise review.
59 training images and 11 validation images are insufficient to establish broad reliability. Inspect near duplicates and incident/source grouping: exact-hash isolation alone does not establish independence. Obtain more verified annotations before claiming generalisation. Unannotated test images are for qualitative inspection only; no test metric is produced.
Artifacts, provenance, library version, args and checksums are in `artifacts/runs/improvements/`. Download before the Colab runtime expires; weights and datasets stay out of Git. No measured improvement is claimed before execution.